# Gold - Fato Atendimento Ocorrências

Registro de tickets de atendimento, ocorrências e taxas de incidentes associados a pedidos de vendas.

In [ ]:
%run ../0_Config/0-Init

In [ ]:
# Parâmetros de Inicialização
table_name = 'fact_ocorrencias'
output_path_data = f"{var_gold}/{table_name}/data"
table_name_schema = f'{var_environment}.{var_gold_schema}.{table_name}'

In [ ]:
from pyspark.sql.functions import col, sha2, date_format, coalesce, lit, when

df_oc = spark.read.table(f"{var_environment}.{var_silver_schema}.case_atendimento_ocorrencias")
df_cab = spark.read.table(f"{var_environment}.{var_silver_schema}.case_erp_pedidos_cabecalho")
df_cli = spark.read.table(f"{var_environment}.{var_gold_schema}.dim_clientes")

df_fact = (
    df_oc
    .join(df_cab, df_oc.id_pedido == df_cab.id_pedido, "left")
    .join(df_cli, df_cab.id_cliente == df_cli.id_cliente, "left")
    
    .withColumn("sk_cliente", coalesce(col("sk_cliente"), sha2(lit("-1"), 256)))
    .withColumn("sk_tempo_ocorrencia", coalesce(date_format(df_oc.data_criacao, "yyyyMMdd").cast("integer"), lit(-1)))
    
    .select(
        col("id_ticket").alias("id_fato_ticket"),
        df_oc.id_pedido.cast("integer").alias("id_pedido"),
        col("sk_cliente"),
        col("sk_tempo_ocorrencia"),
        col("tipo_evento"),
        col("severidade"),
        col("status_ticket")
    )
)

In [ ]:
df_write_fact = df_fact.withColumn("data_key_str", col("sk_tempo_ocorrencia").cast("string"))

import pyspark.sql.functions as F

process_fact(
    df_write=df_write_fact,
    nome_gravacao_tabela=table_name_schema,
    caminho_gravacao_tabela=output_path_data,
    data_formatada="data_key_str",
    chave_clusterby=["sk_tempo_ocorrencia"]
)